In [36]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [37]:
gdp = pd.read_csv("datasets/GDP/gdp_by_country.csv")
hdi = pd.read_csv("datasets/Human_Development_Reports/human_development.csv")

In [38]:
gdp.columns

Index(['Country Name', 'Country Code', 'Year', 'GDP ($USD)', 'Region',
       'Income Group'],
      dtype='str')

In [39]:
hdi.columns

Index(['iso3', 'Year', 'Difference from HDI rank: 2023', 'GDI Group: 2023',
       'GII Rank: 2023', 'HDI Rank: 2023', 'country', 'hdicode', 'region',
       'Human Development Index (value)', 'Life Expectancy at Birth (years)',
       'Expected Years of Schooling (years)',
       'Mean Years of Schooling (years)',
       'Gross National Income Per Capita (2021 PPP$)',
       'Gender Development Index (value)', 'HDI female (value)',
       'Life Expectancy at Birth, female (years)',
       'Expected Years of Schooling, female (years)',
       'Mean Years of Schooling, female (years)',
       'Gross National Income Per Capita, female (2021 PPP$)',
       'HDI male (value)', 'Life Expectancy at Birth, male (years)',
       'Expected Years of Schooling, male (years)',
       'Mean Years of Schooling, male (years)',
       'Gross National Income Per Capita, male (2021 PPP$)',
       'Inequality-adjusted Human Development Index (value)',
       'Coefficient of human inequality', 'Overall lo

In [40]:
# Renaming columns in GDP
gdp = gdp.rename(columns={
    "Country Code": "ISO3",
    "Country Name": "Country",
    "Year": "Year",
    "GDP ($USD)": "GDP",
    "Income Group": "Income_Group",
    "Region": "Region_gdp"
})

# Rename columns in human development
hdi = hdi.rename(columns={
    "iso3": "ISO3",
    "country": "Country_hd",
    "region": "Region",
    "Gender Development Index (value)": "GDI",
    "Population, total (millions)": "Population_millions"
})

# making sure Year is numeric in both
gdp["Year"] = pd.to_numeric(gdp["Year"], errors="coerce")
hdi["Year"] = pd.to_numeric(hdi["Year"], errors="coerce")

# Keeping only needed columns from HDI
hdi_sub = hdi[[
    "ISO3",
    "Year",
    "Country_hd",
    "Region",
    "GDI",
    "Population_millions"
]].copy()

# Merging the two datasets on ISO3 and Year
df = pd.merge(
    gdp,
    hdi_sub,
    on=["ISO3", "Year"],
    how="inner"
)

# Clean
df = df.dropna(subset=["GDP", "GDI"])
df = df[df["GDP"] > 0]

# Check result
print(df.shape)
df.head()

(4919, 10)


,Country,ISO3,Year,GDP,Region_gdp,Income_Group,Country_hd,Region,GDI,Population_millions
2,Albania,ALB,1990,2.028554e+09,Europe & Central Asia,Upper middle income,Albania,ECA,0.938,3.277966
5,Argentina,ARG,1990,1.413527e+11,Latin America & Caribbean,Upper middle income,Argentina,LAC,0.971,32.755901
8,Australia,AUS,1990,3.118407e+11,East Asia & Pacific,High income,Australia,NaN,0.963,17.126298
9,Austria,AUT,1990,1.658114e+11,Europe & Central Asia,High income,Austria,NaN,0.941,7.679625
11,Burundi,BDI,1990,1.132101e+09,Sub-Saharan Africa,Low income,Burundi,SSA,0.822,5.587053


In [41]:
print(df.columns)
print(df[["ISO3", "Year", "Country", "GDP", "Income_Group", "GDI", "Population_millions"]].head())

Index(['Country', 'ISO3', 'Year', 'GDP', 'Region_gdp', 'Income_Group',
       'Country_hd', 'Region', 'GDI', 'Population_millions'],
      dtype='str')
   ISO3  Year    Country           GDP         Income_Group    GDI  \
2   ALB  1990    Albania  2.028554e+09  Upper middle income  0.938   
5   ARG  1990  Argentina  1.413527e+11  Upper middle income  0.971   
8   AUS  1990  Australia  3.118407e+11          High income  0.963   
9   AUT  1990    Austria  1.658114e+11          High income  0.941   
11  BDI  1990    Burundi  1.132101e+09           Low income  0.822   

    Population_millions  
2              3.277966  
5             32.755901  
8             17.126298  
9              7.679625  
11             5.587053  


In [44]:
fig = px.box(
    df_latest,
    x="Income_Group",
    y="GDI",
    color="Income_Group",
    title="Gender Equality Varies Significantly Within Income Groups",
    labels={
        "Income_Group": "Income Group",
        "GDI": "Gender Development Index (GDI)"
    },
    color_discrete_map={
        "Low income": "#e74c3c",
        "Lower middle income": "#f39c12",
        "Upper middle income": "#2ecc71",
        "High income": "#3498db"
    },
    category_orders={
        "Income_Group": [
            "Low income",
            "Lower middle income",
            "Upper middle income",
            "High income"
        ]
    },
    points="outliers"   # cleaner than showing all points
)

fig.update_layout(
    template="simple_white",
    title_x=0.5,
    showlegend=False
)

# cleaner visuals
fig.update_traces(
    marker_size=6,
    opacity=0.8
)

# better scale (VERY important)
fig.update_yaxes(range=[0.75, 1.02])

fig.show()

In [43]:
# ordering the categories for better visualization
trend_df["Income_Group"] = pd.Categorical(
    trend_df["Income_Group"],
    categories=[
        "Low income",
        "Lower middle income",
        "Upper middle income",
        "High income"
    ],
    ordered=True
)

# Create line chart
fig = px.line(
    trend_df,
    x="Year",
    y="GDI",
    color="Income_Group",
    title="Gender Equality Improves Globally, But Income-Based Gaps Persist",
    labels={
        "GDI": "Avg Gender Development Index (GDI)",
        "Year": "Year",
        "Income_Group": "Income Group"
    },
    color_discrete_map={
        "Low income": "#e74c3c",
        "Lower middle income": "#f39c12",
        "Upper middle income": "#2ecc71",
        "High income": "#3498db"
    }
)

fig.update_layout(
    template="simple_white",
    title_x=0.5,
    legend=dict(
        orientation="h",  
        y=-0.2
    )
)

# lines thicker
fig.update_traces(line=dict(width=3))

# Tighten y-axis for focus
fig.update_yaxes(range=[0.75, 1.01])


# Show chart
fig.show()